In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import PolynomialFeatures
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn
import dagshub
import joblib


In [2]:
mlflow.set_tracking_uri('https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow')
dagshub.init(repo_owner='omalbhare', repo_name='medical-insurance-cost-prediction-mlops-project', mlflow=True)
mlflow.set_experiment("Regression Model Comparison")

Accessing as omalbhare

Initialized MLflow to track repo "omalbhare/medical-insurance-cost-prediction-mlops-project"

Repository omalbhare/medical-insurance-cost-prediction-mlops-project initialized!

2025/12/31 17:26:59 INFO mlflow.tracking.fluent: Experiment with name 'Regression Model Comparison' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/079e18bf528641898ac7249ab4a9a41a', creation_time=1767182217653, experiment_id='0', last_update_time=1767182217653, lifecycle_stage='active', name='Regression Model Comparison', tags={}>

In [8]:
X_train = joblib.load("X_train.pkl")
X_test  = joblib.load("X_test.pkl")
y_train = joblib.load("y_train.pkl")
y_test  = joblib.load("y_test.pkl")

In [6]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    y_pred = model.predict(X_test)          # Test predictions
    r2 = r2_score(y_test, y_pred)           # R²: 0.85+ = excellent
    mae = mean_absolute_error(y_test, y_pred) # Avg $ error per prediction
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # $ scale error
    return {"R²": r2, "MAE": mae, "RMSE": rmse}

In [11]:

best_run = {"rmse": float("inf"), "run_id": None}

for degree in [2, 3]:
    with mlflow.start_run(run_name=f"PolynomialRegression_degree_{degree}"):

        poly = PolynomialFeatures(degree=degree)
        X_train_poly = poly.fit_transform(X_train)
        X_test_poly = poly.transform(X_test)

        model = LinearRegression()
        model.fit(X_train_poly, y_train)

        metrics = evaluate_model( model,X_train_poly, X_test_poly, y_train, y_test )

        # Log parameters
        mlflow.log_param("model_name", "PolynomialRegression")
        mlflow.log_param("degree", degree)

        # Log metrics
        mlflow.log_metrics(metrics)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        # Track best model
        if metrics["RMSE"] < best_run["rmse"]:
            best_run = {
                "rmse": metrics["RMSE"],
                "run_id": mlflow.active_run().info.run_id,
                "model": model
            }


2025/12/31 18:16:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run PolynomialRegression_degree_2 at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/07e3a489f8cf452d80d1d54e6c1a25d0
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


2025/12/31 18:16:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run PolynomialRegression_degree_3 at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/2a8f63a27d014fb28c1cbc502c93eebd
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


In [12]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "SVR": SVR()
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):

        model.fit(X_train, y_train)

        metrics = evaluate_model( model,  X_train, X_test,  y_train,  y_test )

        # Log parameters
        mlflow.log_param("model_name", name)

        if hasattr(model, "get_params"):
            for param, value in model.get_params().items():
                mlflow.log_param(param, value)

        # Log metrics
        mlflow.log_metrics(metrics)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        # Track best model
        if metrics["RMSE"] < best_run["rmse"]:
            best_run = {
                "rmse": metrics["RMSE"],
                "run_id": mlflow.active_run().info.run_id,
                "model": model
            }


2025/12/31 18:17:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LinearRegression at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/b3272fcc5a2f47e49d136f1eb5c4612d
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


2025/12/31 18:17:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForest at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/988c7850d0264e37a5f07040dfe19ad6
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


2025/12/31 18:18:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run GradientBoosting at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/52485666aec743b39ac2e32c9a590212
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


2025/12/31 18:19:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/7b9eb937e4254ddd821405dc0da135b7
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0


2025/12/31 18:20:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVR at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0/runs/32aeaa2951a347619a906e8ce748de88
🧪 View experiment at: https://dagshub.com/omalbhare/medical-insurance-cost-prediction-mlops-project.mlflow/#/experiments/0
